# 02 — Limpeza ENEM (2012–2024)

**Objetivo:** processar os microdados brutos do ENEM de cada ano, aplicar filtros de qualidade, criar variáveis derivadas e exportar um arquivo Parquet por ano.

O processamento usa **DuckDB in-memory** para leitura dos CSVs via SQL — atendendo ao requisito da disciplina de usar SQL no pipeline. O DuckDB é muito mais eficiente que o pandas puro para ler arquivos CSV de 5–8 milhões de linhas.

**Inputs:** `datasets/enem/microdados_enem_AAAA/DADOS/MICRODADOS_ENEM_AAAA.csv`  
- Separador: `;` | Encoding: `latin-1`  
- **2016:** nome de arquivo em minúsculo (`microdados_enem_2016.csv`) — tratado automaticamente  
- **2024:** estrutura dividida em dois arquivos separados (ver bloco especial abaixo)

**Output:** `data/processed/enem/enem_AAAA.parquet` — um arquivo por ano (2012–2024)

## 1. Imports e configuração

In [1]:
import duckdb
import pandas as pd
from pathlib import Path

Path('../data/processed/enem').mkdir(parents=True, exist_ok=True)

ANOS = list(range(2012, 2025))


## 2. Constantes e mapeamentos

### Colunas selecionadas
Selecionamos apenas as colunas relevantes para a análise, descartando campos de gabarito, itens de prova e localização de aplicação que não são usados nos modelos.

### Harmonização do Q006 (renda familiar)
O questionário socioeconômico mudou em 2015 — a escala de renda foi reorganizada de 10 para 16 categorias. Para permitir análise temporal comparável, mapeamos ambas as escalas para 5 faixas comuns:
- **A** = sem renda
- **B** = até 1 salário mínimo  
- **C** = 1 a 2 salários mínimos  
- **D** = 2 a 5 salários mínimos  
- **E** = acima de 5 salários mínimos

### Mapeamento de macrorregiões
Derivamos a macrorregião diretamente da sigla UF da escola (`SG_UF_ESC`), que serve como proxy para o estado de residência do candidato.

### Faixas de nota
Bins: `[0, 400, 500, 600, 700, 800, 1001]` — mesmas faixas para CN, CH, LC, MT e Redação, usadas como variável alvo nos modelos de classificação.

In [2]:
COLUNAS = [
    'NU_ANO', 'CO_MUNICIPIO_ESC', 'SG_UF_ESC',
    'TP_SEXO', 'TP_COR_RACA', 'TP_ESCOLA',
    'TP_PRESENCA_CN', 'TP_PRESENCA_CH', 'TP_PRESENCA_LC', 'TP_PRESENCA_MT',
    'Q001', 'Q002', 'Q006',
    'NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO'
]

# Colunas Q estendidas — disponíveis nos CSVs de 2019–2023 com semântica estável
COLUNAS_Q_EXT = ['Q003', 'Q004', 'Q005', 'Q007', 'Q010', 'Q011', 'Q012',
                 'Q014', 'Q017', 'Q018', 'Q019', 'Q020', 'Q022', 'Q024']

Q006_2012_2014 = {'A':'A','B':'B','C':'C','D':'D','E':'D','F':'E','G':'E','H':'E','I':'E','J':'E'}
Q006_2015_2023 = {'A':'A','B':'B','C':'C','D':'C','E':'D','F':'D','G':'D','H':'D',
                  'I':'E','J':'E','K':'E','L':'E','M':'E','N':'E','O':'E','P':'E'}

SG_UF_REGIAO = {
    'RO':'Norte','AC':'Norte','AM':'Norte','RR':'Norte','PA':'Norte','AP':'Norte','TO':'Norte',
    'MA':'Nordeste','PI':'Nordeste','CE':'Nordeste','RN':'Nordeste','PB':'Nordeste',
    'PE':'Nordeste','AL':'Nordeste','SE':'Nordeste','BA':'Nordeste',
    'MG':'Sudeste','ES':'Sudeste','RJ':'Sudeste','SP':'Sudeste',
    'PR':'Sul','SC':'Sul','RS':'Sul',
    'MS':'Centro-Oeste','MT':'Centro-Oeste','GO':'Centro-Oeste','DF':'Centro-Oeste'
}

BINS   = [0, 400, 500, 600, 700, 800, 1001]
LABELS = ['<400','400-500','500-600','600-700','700-800','>800']
AREAS  = ['CN','CH','LC','MT','REDACAO']
COLS_STR = ['TP_SEXO','TP_COR_RACA','TP_ESCOLA','SG_UF_ESC',
            'Q001','Q002','Q006','Q006_HARM','REGIAO',
            'FAIXA_CN','FAIXA_CH','FAIXA_LC','FAIXA_MT','FAIXA_REDACAO',
            # Q colunas brutas carregadas dos CSVs (2019+)
            'Q003','Q004','Q007','Q010','Q011','Q012','Q014','Q017','Q018','Q019','Q020','Q022','Q024',
            # Q colunas harmonizadas (nome final no parquet)
            'Q_OCUP_PAI','Q_OCUP_MAE','Q_N_PESSOAS','Q_EMPREGADA',
            'Q_CARRO','Q_MOTO','Q_GELADEIRA','Q_LAVADORA','Q_TV',
            'Q_COMPUTADOR','Q_INTERNET','Q_CELULAR','Q_TIPO_ESCOLA_EM','Q_BOLSA_FAM']


## 3. Loop de processamento por ano

### Filtro de presença
Mantemos apenas candidatos que fizeram **todos os quatro dias de prova** e têm nota de Matemática válida:
```sql
WHERE TP_PRESENCA_CN = 1 AND TP_PRESENCA_CH = 1
  AND TP_PRESENCA_LC = 1 AND TP_PRESENCA_MT = 1
  AND NU_NOTA_MT IS NOT NULL
```
Isso remove em média ~33% das inscrições (55% em 2020 por conta do COVID-19).

### Tratamento especial 2024
O INEP separou os microdados de 2024 em dois arquivos:
- `PARTICIPANTES_2024.csv`: dados demográficos e questionário (Q006 etc.)
- `RESULTADOS_2024.csv`: escola, presença e notas

Como não há chave de join explícita entre eles, o join é feito por **posição de linha** via `ROW_NUMBER() OVER ()` — os dois arquivos têm exatamente 4.332.944 linhas na mesma ordem de inscrição.

O `TP_ESCOLA` de 2024 é derivado de `TP_DEPENDENCIA_ADM_ESC`:  
`4 (privada) → 3` | `demais (federal/estadual/municipal) → 2`

### Padronização de tipos
Todas as colunas categóricas são forçadas para `str` antes de salvar — necessário porque `TP_SEXO` era inteiro (`0`/`1`) em 2012–2014 e string (`'M'`/`'F'`) de 2015 em diante. Sem isso, o `read_parquet(..., union_by_name=true)` no notebook 04 falharia com erro de tipo.

In [3]:
def get_csv_path(ano):
    base = f'../datasets/enem/microdados_enem_{ano}/DADOS/'
    upper = f'{base}MICRODADOS_ENEM_{ano}.csv'
    lower = f'{base}microdados_enem_{ano}.csv'
    return upper if Path(upper).exists() else lower

con = duckdb.connect()  # in-memory
resumo = []

for ano in ANOS:
    if ano == 2024:
        base   = '../datasets/enem/microdados_enem_2024/DADOS/'
        p_path = f'{base}PARTICIPANTES_2024.csv'
        r_path = f'{base}RESULTADOS_2024.csv'
        df = con.execute(f"""
            WITH p AS (
                SELECT ROW_NUMBER() OVER () AS rn,
                       NU_ANO, TP_SEXO, TP_COR_RACA, Q001, Q002,
                       -- Em 2024: Q006=binário (sem/com renda), Q007=escala de renda (como Q006 de 2015-2023)
                       -- Para Q006_HARM, usamos Q007 do CSV como Q006
                       p.Q007 AS Q006,
                       -- Features socioeconômicas estendidas com remapeamento de numeração 2024
                       p.Q003, p.Q004, CAST(p.Q005 AS VARCHAR) AS Q005,
                       p.Q008 AS Q007,   -- empregada doméstica (era Q007 em 2019-2023)
                       p.Q011 AS Q010,   -- carro (era Q010)
                       p.Q012 AS Q011,   -- moto (era Q011)
                       p.Q013 AS Q012,   -- geladeira (era Q012)
                       p.Q015 AS Q014,   -- lavadora (era Q014)
                       p.Q018 AS Q017,   -- tv (era Q017)
                       p.Q021 AS Q018,   -- computador (era Q018)
                       p.Q020 AS Q019,   -- internet (era Q019)
                       p.Q022 AS Q020,   -- celular (era Q020)
                       p.Q023 AS Q022    -- tipo escola EM (era Q022)
                FROM read_csv_auto('{p_path}', delim=';', header=true, ignore_errors=true) p
            ),
            r AS (
                SELECT ROW_NUMBER() OVER () AS rn,
                       CO_MUNICIPIO_ESC, SG_UF_ESC, TP_DEPENDENCIA_ADM_ESC,
                       TP_PRESENCA_CN, TP_PRESENCA_CH, TP_PRESENCA_LC, TP_PRESENCA_MT,
                       NU_NOTA_CN, NU_NOTA_CH, NU_NOTA_LC, NU_NOTA_MT, NU_NOTA_REDACAO
                FROM read_csv_auto('{r_path}', delim=';', header=true, ignore_errors=true)
            )
            SELECT p.NU_ANO, r.CO_MUNICIPIO_ESC, r.SG_UF_ESC,
                   p.TP_SEXO, p.TP_COR_RACA,
                   CASE r.TP_DEPENDENCIA_ADM_ESC WHEN 4 THEN 3 ELSE 2 END AS TP_ESCOLA,
                   r.TP_PRESENCA_CN, r.TP_PRESENCA_CH, r.TP_PRESENCA_LC, r.TP_PRESENCA_MT,
                   p.Q001, p.Q002, p.Q006,
                   p.Q003, p.Q004, p.Q005,
                   p.Q007, p.Q010, p.Q011, p.Q012, p.Q014,
                   p.Q017, p.Q018, p.Q019, p.Q020, p.Q022,
                   r.NU_NOTA_CN, r.NU_NOTA_CH, r.NU_NOTA_LC, r.NU_NOTA_MT, r.NU_NOTA_REDACAO
            FROM p JOIN r ON p.rn = r.rn
            WHERE r.TP_PRESENCA_CN=1 AND r.TP_PRESENCA_CH=1
              AND r.TP_PRESENCA_LC=1 AND r.TP_PRESENCA_MT=1
              AND r.NU_NOTA_MT IS NOT NULL
        """).df()
        total_antes = con.execute(
            f"SELECT COUNT(*) FROM read_csv_auto('{r_path}', delim=';', header=true, ignore_errors=true)"
        ).fetchone()[0]

        # Para 2024: colunas harmonizadas diretas (Q já remapeadas no SQL acima)
        df['Q_OCUP_PAI']       = df.get('Q003')
        df['Q_OCUP_MAE']       = df.get('Q004')
        df['Q_N_PESSOAS']      = df.get('Q005')
        df['Q_EMPREGADA']      = df.get('Q007')   # já remapeado: Q008 do CSV
        df['Q_CARRO']          = df.get('Q010')   # já remapeado: Q011 do CSV
        df['Q_MOTO']           = df.get('Q011')   # já remapeado: Q012 do CSV
        df['Q_GELADEIRA']      = df.get('Q012')   # já remapeado: Q013 do CSV
        df['Q_LAVADORA']       = df.get('Q014')   # já remapeado: Q015 do CSV
        df['Q_TV']             = df.get('Q017')   # já remapeado: Q018 do CSV
        df['Q_COMPUTADOR']     = df.get('Q018')   # já remapeado: Q021 do CSV
        df['Q_INTERNET']       = df.get('Q019')   # já remapeado: Q020 do CSV
        df['Q_CELULAR']        = df.get('Q020')   # já remapeado: Q022 do CSV
        df['Q_TIPO_ESCOLA_EM'] = df.get('Q022')   # já remapeado: Q023 do CSV
        df['Q_BOLSA_FAM']      = None              # Q024 não existe em 2024

    else:
        csv = get_csv_path(ano)

        if ano >= 2019:
            # Para 2019-2023: inclui as colunas Q estendidas que têm semântica estável
            extra_cols = ', '.join(COLUNAS_Q_EXT)
            cols_sql = ', '.join(COLUNAS) + ', ' + extra_cols
        else:
            # Para 2012-2018: só as colunas base (Q003-Q024 existem mas com significado diferente)
            cols_sql = ', '.join(COLUNAS)

        df = con.execute(f"""
            SELECT {cols_sql}
            FROM read_csv_auto('{csv}', delim=';', header=true, ignore_errors=true)
            WHERE TP_PRESENCA_CN=1 AND TP_PRESENCA_CH=1
              AND TP_PRESENCA_LC=1 AND TP_PRESENCA_MT=1
              AND NU_NOTA_MT IS NOT NULL
        """).df()
        total_antes = con.execute(
            f"SELECT COUNT(*) FROM read_csv_auto('{csv}', delim=';', header=true, ignore_errors=true)"
        ).fetchone()[0]

        # Cria colunas harmonizadas Q_*
        if ano >= 2019:
            df['Q_OCUP_PAI']       = df.get('Q003', pd.Series(dtype=str))
            df['Q_OCUP_MAE']       = df.get('Q004', pd.Series(dtype=str))
            df['Q_N_PESSOAS']      = df.get('Q005', pd.Series(dtype=str))
            df['Q_EMPREGADA']      = df.get('Q007', pd.Series(dtype=str))
            df['Q_CARRO']          = df.get('Q010', pd.Series(dtype=str))
            df['Q_MOTO']           = df.get('Q011', pd.Series(dtype=str))
            df['Q_GELADEIRA']      = df.get('Q012', pd.Series(dtype=str))
            df['Q_LAVADORA']       = df.get('Q014', pd.Series(dtype=str))
            df['Q_TV']             = df.get('Q017', pd.Series(dtype=str))
            df['Q_COMPUTADOR']     = df.get('Q018', pd.Series(dtype=str))
            df['Q_INTERNET']       = df.get('Q019', pd.Series(dtype=str))
            df['Q_CELULAR']        = df.get('Q020', pd.Series(dtype=str))
            df['Q_TIPO_ESCOLA_EM'] = df.get('Q022', pd.Series(dtype=str))
            df['Q_BOLSA_FAM']      = df.get('Q024', pd.Series(dtype=str))
        else:
            # 2012-2018: questões com numeração semanticamente incompatível — deixa NULL
            for col in ['Q_OCUP_PAI','Q_OCUP_MAE','Q_N_PESSOAS','Q_EMPREGADA',
                        'Q_CARRO','Q_MOTO','Q_GELADEIRA','Q_LAVADORA','Q_TV',
                        'Q_COMPUTADOR','Q_INTERNET','Q_CELULAR','Q_TIPO_ESCOLA_EM','Q_BOLSA_FAM']:
                df[col] = None

    for area in AREAS:
        df[f'FAIXA_{area}'] = pd.cut(df[f'NU_NOTA_{area}'], bins=BINS, labels=LABELS, right=False)

    mapa = Q006_2012_2014 if ano <= 2014 else Q006_2015_2023
    df['Q006_HARM'] = df['Q006'].map(mapa)
    df['REGIAO']    = df['SG_UF_ESC'].map(SG_UF_REGIAO)

    for col in COLS_STR:
        if col in df.columns:
            df[col] = df[col].astype(str).where(df[col].notna(), None)

    df.to_parquet(f'../data/processed/enem/enem_{ano}.parquet', index=False)
    resumo.append({'ano': ano, 'total_bruto': total_antes,
                   'apos_filtro': len(df), 'removidos': total_antes - len(df)})
    print(f'{ano}: {total_antes:,} -> {len(df):,} ({total_antes-len(df):,} removidos)')

con.close()
pd.DataFrame(resumo)


2012: 5,791,065 -> 4,079,886 (1,711,179 removidos)
2013: 7,173,563 -> 5,007,934 (2,165,629 removidos)
2014: 8,722,248 -> 5,947,909 (2,774,339 removidos)
2015: 7,746,427 -> 5,604,905 (2,141,522 removidos)
2016: 8,627,179 -> 5,818,264 (2,808,915 removidos)
2017: 6,731,278 -> 4,426,692 (2,304,586 removidos)
2018: 5,513,733 -> 3,893,729 (1,620,004 removidos)
2019: 5,095,171 -> 3,701,910 (1,393,261 removidos)
2020: 5,783,109 -> 2,588,681 (3,194,428 removidos)
2021: 3,389,832 -> 2,238,107 (1,151,725 removidos)
2022: 3,476,105 -> 2,344,823 (1,131,282 removidos)
2023: 3,933,955 -> 2,678,264 (1,255,691 removidos)
2024: 4,332,944 -> 2,990,093 (1,342,851 removidos)


,ano,total_bruto,apos_filtro,removidos
0,2012,5791065,4079886,1711179
1,2013,7173563,5007934,2165629
2,2014,8722248,5947909,2774339
3,2015,7746427,5604905,2141522
4,2016,8627179,5818264,2808915
5,2017,6731278,4426692,2304586
6,2018,5513733,3893729,1620004
7,2019,5095171,3701910,1393261
8,2020,5783109,2588681,3194428
9,2021,3389832,2238107,1151725


## 4. Verificação Q4 — Candidatos por ano (query SQL obrigatória)

Lê todos os parquets gerados com DuckDB in-memory para verificar a contagem por ano após o filtro de presença.  
Esta é a **Query Q4** exigida pela disciplina.

In [4]:
con2 = duckdb.connect()
print(con2.execute("""
    SELECT NU_ANO, COUNT(*) AS candidatos
    FROM read_parquet('../data/processed/enem/enem_*.parquet', union_by_name=true)
    GROUP BY NU_ANO ORDER BY NU_ANO
""").df().to_string(index=False))
con2.close()


 NU_ANO  candidatos
   2012     4079886
   2013     5007934
   2014     5947909
   2015     5604905
   2016     5818264
   2017     4426692
   2018     3893729
   2019     3701910
   2020     2588681
   2021     2238107
   2022     2344823
   2023     2678264
   2024     2990093
